# Loading and Exploring Raw EEG Data
Dataset: ERP CORE (Kappenman et al., 2021), one example subject, provided by MNE-Python.

In [1]:
import mne

data_path = mne.datasets.erp_core.data_path()
print(data_path)

for f in sorted(data_path.glob("*")):
    print(f.name)

C:\Users\Lenovo\mne_data\MNE-ERP-CORE-data
ERP-CORE_Subject-001_Task-Flankers_eeg.fif
README.txt
version.txt


In [2]:
raw_file = data_path / "ERP-CORE_Subject-001_Task-Flankers_eeg.fif"
raw = mne.io.read_raw_fif(raw_file, preload=True)
raw

Opening raw data file C:\Users\Lenovo\mne_data\MNE-ERP-CORE-data\ERP-CORE_Subject-001_Task-Flankers_eeg.fif...


    Range : 0 ... 935935 =      0.000 ...   913.999 secs
Ready.
Reading 0 ... 935935  =      0.000 ...   913.999 secs...


<Raw | ERP-CORE_Subject-001_Task-Flankers_eeg.fif, 33 x 935936 (914.0 s), ~235.7 MiB, data loaded>

## Task Design: Flanker Task (ERP CORE)

This notebook uses data from one participant of the ERP CORE dataset
(Kappenman et al., 2021), recorded during an arrow version of the
Eriksen flanker task.

The task consisted of 400 trials in a 2 × 2 within-subject design:

- **Congruency:** congruent (e.g., `<<<<<`) vs. incongruent (e.g., `<<><<`)
- **Target direction:** left vs. right

Each of the four conditions contained 100 trials. The participant responded
to the direction of the central arrow with a left- or right-hand button press,
ignoring the flanking arrows. In the event annotations, congruent and
incongruent trials are labeled `compatible` and `incompatible`, respectively.

The recorded responses (n = 402) will be used to examine two ERP components:
the error-related negativity (ERN) and the lateralized readiness potential (LRP).

In [3]:
from collections import Counter

print(raw.ch_names)
print(raw.annotations)
print(Counter(raw.annotations.description))

['FP1', 'F3', 'F7', 'FC3', 'C3', 'C5', 'P3', 'P7', 'P9', 'PO7', 'PO3', 'O1', 'Oz', 'Pz', 'CPz', 'FP2', 'Fz', 'F4', 'F8', 'FC4', 'FCz', 'Cz', 'C4', 'C6', 'P4', 'P8', 'P10', 'PO8', 'PO4', 'O2', 'HEOG_left', 'HEOG_right', 'VEOG_lower']
<Annotations | 802 segments: response/left (202), response/right (200), ...>
Counter({'response/left': 202, 'response/right': 200, 'stimulus/compatible/target_left': 100, 'stimulus/incompatible/target_right': 100, 'stimulus/incompatible/target_left': 100, 'stimulus/compatible/target_right': 100})


In [4]:
events, event_id = mne.events_from_annotations(raw)
print(event_id)
print(events[:10])

Used Annotations descriptions: ['response/left', 'response/right', 'stimulus/compatible/target_left', 'stimulus/compatible/target_right', 'stimulus/incompatible/target_left', 'stimulus/incompatible/target_right']
{'response/left': 1, 'response/right': 2, 'stimulus/compatible/target_left': 3, 'stimulus/compatible/target_right': 4, 'stimulus/incompatible/target_left': 5, 'stimulus/incompatible/target_right': 6}
[[20101     0     2]
 [30243     0     2]
 [34675     0     3]
 [35240     0     1]
 [36261     0     3]
 [36706     0     1]
 [37848     0     6]
 [38369     0     2]
 [39401     0     3]
 [39917     0     1]]


In [5]:
sfreq = raw.info["sfreq"]
rows = []

for i in range(len(events) - 1):
    sample, _, code = events[i]
    next_sample, _, next_code = events[i + 1]

    is_stimulus = code in [3, 4, 5, 6]
    is_response = next_code in [1, 2]
    if not (is_stimulus and is_response):
        continue

    rt_samples = next_sample - sample          
    rt_ms = (rt_samples / sfreq) * 1000

    if code in [3, 4]:
        congruency = 'congruent'
    else:  
        congruency = 'incongruent'

    if code in [3, 5]:
        stimulus_direction = 'left'
        correct_response = 1  
    else: 
        stimulus_direction = 'right'
        correct_response = 2  
    
    
    is_correct = (next_code == correct_response)
    
    
    rows.append({
        'stimulus_sample': sample,
        'response_sample': next_sample,
        'stimulus_code': code,
        'response_code': next_code,
        'rt_ms': rt_ms,
        'congruency': congruency,
        'direction': stimulus_direction,
        'correct': is_correct
    })


In [6]:
import pandas as pd
df = pd.DataFrame(rows)
print(len(df))
print(df["correct"].value_counts())
df.head()

400
correct
True     346
False     54
Name: count, dtype: int64


,stimulus_sample,response_sample,stimulus_code,response_code,rt_ms,congruency,direction,correct
0,34675,35240,3,1,551.757812,congruent,left,True
1,36261,36706,3,1,434.570312,congruent,left,True
2,37848,38369,6,2,508.789062,incongruent,right,True
3,39401,39917,3,1,503.906250,congruent,left,True
4,40885,41441,5,1,542.968750,incongruent,left,True


In [7]:
accuracy = df.groupby("congruency")["correct"].mean() * 100
print(accuracy.round(1))

rt = df[df["correct"]].groupby("congruency")["rt_ms"].agg(["mean", "median"])
print(rt.round(1))

congruency
congruent      96.5
incongruent    76.5
Name: correct, dtype: float64
              mean  median
congruency                
congruent    392.8   382.8
incongruent  454.4   447.3


In [8]:
incong = df[df["congruency"] == "incongruent"]
print(incong.groupby("correct")["rt_ms"].agg(["count", "median"]).round(1))

         count  median
correct               
False       47   339.8
True       153   447.3


In [9]:
df.to_csv("../results/behavior_flanker_sub-001.csv", index=False)

## Behavioral Results

Overall accuracy was 86.5% (346/400 trials). Accuracy was higher for
congruent (96.5%) than for incongruent trials (76.5%), corresponding to
7 and 47 errors, respectively.

Reaction times (RTs) were analyzed for correct trials only. Because RT
distributions are right-skewed, both means and medians are reported.
Responses were slower on incongruent than on congruent trials
(median: 447.3 vs. 382.8 ms; mean: 454.4 vs. 392.8 ms), yielding a
conflict effect of approximately 65 ms (median difference).

Within incongruent trials, errors were markedly faster than correct
responses (median: 339.8 ms, n = 47 vs. 447.3 ms, n = 153).

**Interpretation.** These results reproduce the classic flanker pattern.
Early in processing, the flanking arrows activate the response they point
to, which on incongruent trials is the wrong one. Fast responses, made
before attention has fully narrowed onto the central target, are therefore
more likely to be errors.

**Implications for the ERN analysis.** Because 47 of the 54 errors occurred
on incongruent trials, comparing error trials with all correct trials would
confound response accuracy with congruency. The ERN will therefore be
assessed by comparing errors and correct responses within the incongruent
condition.
